# Retrain YOLOv8n / YOLOv8s on the cleaned five-crop dataset

1. On your PC, zip the cleaned dataset **folder contents** (`images/`, `labels/`) as `dataset_clean.zip` and upload it to `MyDrive/LDD/`.
2. Runtime → Change runtime type → **T4 GPU**. Run all cells.
3. Checkpoints go to `MyDrive/LDD/runs/` every 5 epochs. If Colab disconnects, just run all cells again: training resumes from `last.pt`.
4. Download `MyDrive/LDD/runs/v8n_clean/weights/best.pt` (and `v8s_clean` if trained) into the project's `weights/` folder and run `experiments/run_all.py` locally.

Hyperparameters match `train.py` and the original run (imgsz 640, batch 16, seed 0, same augmentation).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install ultralytics
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, zipfile
DRIVE   = '/content/drive/MyDrive/LDD'
DATA    = '/content/dataset'          # local disk: much faster than reading from Drive
PROJECT = f'{DRIVE}/runs'             # on Drive: survives session resets

if not os.path.exists(f'{DATA}/images/train'):
    with zipfile.ZipFile(f'{DRIVE}/dataset_clean.zip') as z:
        z.extractall(DATA)
for s in ('train', 'val', 'test'):
    print(s, len(os.listdir(f'{DATA}/images/{s}')))

YAML = '/content/crop_dataset.yaml'
open(YAML, 'w').write(f'''path: {DATA}
train: images/train
val: images/val
test: images/test

nc: 5
names:
  0: sugarcane
  1: wheat
  2: maize
  3: rice
  4: soybean
''')

In [ ]:
from ultralytics import YOLO

def train(size, name, epochs=100):
    last = f'{PROJECT}/{name}/weights/last.pt'
    if os.path.exists(last):
        print('Resuming', last)
        return YOLO(last).train(resume=True)
    return YOLO(f'yolov8{size}.pt').train(
        data=YAML, epochs=epochs, imgsz=640, batch=16, device=0,
        project=PROJECT, name=name, exist_ok=True, save_period=5, seed=0,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, flipud=0.5, fliplr=0.5,
        mosaic=1.0, mixup=0.1, copy_paste=0.1,
        degrees=10.0, translate=0.1, scale=0.5, shear=2.0)

train('n', 'v8n_clean')

## Optional: YOLOv8s for the n-vs-s ablation (same settings)
Run this only if GPU time allows; it resumes the same way.

In [ ]:
train('s', 'v8s_clean')

In [ ]:
# Every run keeps results.csv (per-epoch metrics) and results.png on Drive
!ls -la {PROJECT}/*/weights {PROJECT}/*/results.csv {PROJECT}/*/results.png